# Bronze → Silver: salestrack_sales_final

**Propósito:** Leer `dbo.salestrack_sales_final` del lakehouse Bronze, aplicar transformaciones de calidad y escribir en el lakehouse Silver de forma idempotente.

**Transformaciones aplicadas:**
- `date`: cast a `DateType` (eliminar componente hora)
- `original_sellout` y `percentage`: cast a `DoubleType` (vienen como string en el origen)
- Renombrado de `Agrup1` → `agrup1`, `Agrup2` → `agrup2` (normalización de naming)
- `dq_flag_unidades`: columna de calidad de dato — marca filas donde `original_unidades` tiene valores anómalos (> umbral, probablemente error de origen)
- Trim de columnas string
- Columna de auditoría `_silver_load_ts`

**Nota sobre `original_unidades`:** Se han detectado valores en notación científica (ej. 3.8e+19) en el origen. Se mantiene la columna tal cual pero se añade un flag de calidad para trazabilidad.

**Idempotencia:** `overwrite` + `overwriteSchema=true`. Seguro de re-ejecutar.

In [ ]:
%run ./config

StatementMeta(, 2907428b-6439-4544-a16b-0cf6cbd3e7c6, 3, Finished, Available, Finished, True)

Config cargada → Bronze: lakehouse_poc | Silver: silver_lakehouse_poc | Gold: Gold


In [ ]:


BRONZE_TABLE = f"{DEFAULT_SCHEMA}.salestrack_sales_final"
SILVER_TABLE = f"{DEFAULT_SCHEMA}.salestrack_sales_final"

StatementMeta(, 2907428b-6439-4544-a16b-0cf6cbd3e7c6, 4, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DoubleType
from datetime import datetime

df_bronze = spark.read.table(f"`{BRONZE_LAKEHOUSE}`.{BRONZE_TABLE}")

print(f"Filas leídas desde Bronze: {df_bronze.count()}")
df_bronze.printSchema()

StatementMeta(, 2907428b-6439-4544-a16b-0cf6cbd3e7c6, 5, Finished, Available, Finished, False)

Filas leídas desde Bronze: 304041
root
 |-- date: date (nullable = true)
 |-- yearmonth: string (nullable = true)
 |-- salestrack_group: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- delegate_id: string (nullable = true)
 |-- delegate_name: string (nullable = true)
 |-- line: string (nullable = true)
 |-- province_id: string (nullable = true)
 |-- province_name: string (nullable = true)
 |-- Agrup1: string (nullable = true)
 |-- Agrup2: string (nullable = true)
 |-- original_sellout: double (nullable = true)
 |-- sell_out: double (nullable = true)
 |-- original_unidades: decimal(38,18) (nullable = true)
 |-- unidades: double (nullable = true)
 |-- percentage: double (nullable = true)



In [ ]:
# ─── TRANSFORMACIONES ─────────────────────────────────────────────────────────

df_silver = (
    df_bronze

    # 1. Cast date → DateType
    .withColumn("date", F.col("date").cast(DateType()))

    # 2. Cast columnas que vienen como string en el origen
    .withColumn("original_sellout", F.col("original_sellout").cast(DoubleType()))
    .withColumn("percentage",       F.col("percentage").cast(DoubleType()))

    # 3. Normalización de naming (Agrup1/Agrup2 con mayúscula inicial → minúsculas)
    .withColumnRenamed("Agrup1", "agrup1")
    .withColumnRenamed("Agrup2", "agrup2")

    # 4. Flag de calidad de dato para original_unidades
    #    True = valor sospechoso (posible error de origen)
    .withColumn(
        "dq_flag_unidades_anomaly",
        F.when(
            F.col("original_unidades").isNotNull() &
            (F.abs(F.col("original_unidades")) > UNIDADES_ANOMALY_THRESHOLD),
            F.lit(True)
        ).otherwise(F.lit(False))
    )

    # 5. Trim de strings
    .withColumn("salestrack_group", F.trim(F.col("salestrack_group")))
    .withColumn("customer_name",    F.trim(F.col("customer_name")))
    .withColumn("delegate_name",    F.trim(F.col("delegate_name")))
    .withColumn("line",             F.trim(F.col("line")))
    .withColumn("province_name",    F.trim(F.col("province_name")))
    .withColumn("agrup1",           F.trim(F.col("agrup1")))
    .withColumn("agrup2",           F.trim(F.col("agrup2")))

    # 6. Auditoría
    .withColumn("_silver_load_ts", F.lit(datetime.utcnow().isoformat()).cast("timestamp"))
)

StatementMeta(, 2907428b-6439-4544-a16b-0cf6cbd3e7c6, 6, Finished, Available, Finished, False)

In [ ]:
from pyspark.sql import functions as F

# ─── LIMPIEZA CUSTOMER_ID ───────────────────────────────────────────────
df_silver = df_silver.filter(
    F.col("customer_id").isNotNull() & (F.trim(F.col("customer_id")) != "")
)

# ─── VALIDACIÓN ─────────────────────────────────────────────────────────
row_count          = df_silver.count()
null_dates         = df_silver.filter(F.col("date").isNull()).count()
null_sellout       = df_silver.filter(F.col("original_sellout").isNull()).count()
anomaly_unidades   = df_silver.filter(F.col("dq_flag_unidades_anomaly") == True).count()

print(f"Filas a escribir en Silver       : {row_count}")
print(f"Filas con date nulo              : {null_dates}")
print(f"Filas con original_sellout nulo  : {null_sellout}")
print(f"Filas con unidades anómalas (DQ) : {anomaly_unidades}")

assert null_dates == 0, "ERROR: hay fechas nulas en salestrack_sales_final"

if anomaly_unidades > 0:
    print(f"⚠ AVISO: {anomaly_unidades} filas con unidades anómalas.")

df_silver.show(3)

StatementMeta(, 2907428b-6439-4544-a16b-0cf6cbd3e7c6, 7, Finished, Available, Finished, False)

Filas a escribir en Silver       : 261239
Filas con date nulo              : 0
Filas con original_sellout nulo  : 0
Filas con unidades anómalas (DQ) : 0
+----------+---------+----------------+-----------+--------------------+-----------+-------------+----+-----------+-------------+------+------+----------------+--------+--------------------+--------+----------+------------------------+--------------------+
|      date|yearmonth|salestrack_group|customer_id|       customer_name|delegate_id|delegate_name|line|province_id|province_name|agrup1|agrup2|original_sellout|sell_out|   original_unidades|unidades|percentage|dq_flag_unidades_anomaly|     _silver_load_ts|
+----------+---------+----------------+-----------+--------------------+-----------+-------------+----+-----------+-------------+------+------+----------------+--------+--------------------+--------+----------+------------------------+--------------------+
|2025-11-01|   202511|  PEDIDO DIRECTO|        935|HOSPITAL ESPERIT ...|    

In [ ]:
# ─── ESCRITURA IDEMPOTENTE EN SILVER ──────────────────────────────────────────
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"`{SILVER_LAKEHOUSE}`.{SILVER_TABLE}")
)

print(f"✓ Tabla {SILVER_LAKEHOUSE}.{SILVER_TABLE} escrita correctamente con {row_count} filas.")

StatementMeta(, 2907428b-6439-4544-a16b-0cf6cbd3e7c6, 8, Finished, Available, Finished, False)

✓ Tabla silver_lakehouse_poc.dbo.salestrack_sales_final escrita correctamente con 261239 filas.
